In [ ]:
import numpy as np
import time

# ===============================
# ESN PARAMETERS (UNCHANGED)
# ===============================
N_RESERVOIR = 1024
N_INPUT = 3
LEAK_RATE = 0.3

N_OUTPUT_BITS = 256
N_STM_BITS = 32
STM_P = 0.4

TAPPED_NEURONS = np.arange(0, N_RESERVOIR, N_RESERVOIR // N_OUTPUT_BITS)

# ===============================
# LOAD TRAINED WEIGHTS
# ===============================
print("Loading weights...")
W_in  = np.load("W_in_canonical.npy")
W_res = np.load("W_res_canonical.npy")
W_out = np.load("W_out_chua_final.npy")
print("Weights loaded.")

# ===============================
# ACTIVATION
# ===============================
def activation_func(x):
    return np.tanh(x)

# ===============================
# SRT + ARX
# ===============================
def extract_raw_bits_srt(r):
    tapped = r[TAPPED_NEURONS]
    shifted = np.floor(tapped * 65536).astype(np.int64)
    return (shifted & 1).astype(np.uint8)

def arx_mix(raw_bits):
    # Pack 256 bits → 32 bytes
    packed = np.packbits(raw_bits)

    # Reverse + make contiguous
    packed = packed[::-1].copy()

    # Interpret as 4x uint64
    words = packed.view(np.uint64)
    a, b, c, d = [int(x) for x in words[:4]]  # convert to Python int

    MASK = (1 << 64) - 1

    def rot(x, n):
        return ((x << n) | (x >> (64 - n))) & MASK

    # ---- ChaCha-style ARX ----
    a = (a + b) & MASK
    d ^= a; d = rot(d, 32)

    c = (c + d) & MASK
    b ^= c; b = rot(b, 16)

    a = (a + b) & MASK
    d ^= a; d = rot(d, 8)

    c = (c + d) & MASK
    b ^= c; b = rot(b, 7)

    out = np.array([a, b, c, d], dtype=np.uint64)

    # Convert back to 256 bits
    return np.unpackbits(out.view(np.uint8))[::-1][:256]



# ===============================
# SKEW TENT MAP
# ===============================
def skew_tent_map(x, p):
    if x < p:
        return x / p
    else:
        return (1 - x) / (1 - p)

def extract_stm_bits(x):
    shifted = int(np.floor(x * (2**32)))
    return np.unpackbits(
        np.array([shifted], dtype=np.uint32).view(np.uint8)
    )[::-1][:32]

# ===============================
# MAIN GENERATOR
# ===============================
def generate_bits(n_steps=5000, warmup=1000):
    r = np.zeros(N_RESERVOIR)
    u = np.array([0.1, 0.2, 1.0])
    x_stm = 0.5

    output = []

    start = time.time()
    for i in range(n_steps + warmup):
        r = (1 - LEAK_RATE) * r + LEAK_RATE * activation_func(W_res @ r + W_in @ u)
        u = W_out @ r

        if i < warmup:
            continue

        bits_A = arx_mix(extract_raw_bits_srt(r))

        y_scaled = np.clip((u[0] + 15.0) / 30.0, 0.0, 1.0)
        x_stm = skew_tent_map(y_scaled, STM_P)
        bits_B = extract_stm_bits(x_stm)

        blocks = np.split(bits_A, 8)
        prev = bits_B
        out = []

        for blk in blocks:
            blk = blk ^ prev
            out.append(blk)
            prev = blk

        output.append(np.concatenate(out))

        if (i % 500) == 0:
            print(f"Step {i}")

    print("Generation time:", time.time() - start)
    return np.concatenate(output)

# ===============================
# RUN + SAVE
# ===============================
print("Generating CSPRNG bitstream...")
bits = generate_bits()

packed = np.packbits(bits)
with open("hybrid_csprng_output.bin", "wb") as f:
    f.write(packed.tobytes())

print("DONE.")
print("Total bits:", len(bits))
print("File: hybrid_csprng_output.bin")


Loading weights...
Weights loaded.
Generating CSPRNG bitstream...
Step 1000
Step 1500
Step 2000
Step 2500
Step 3000
